# Configure and publish the Fabric data agent

Points `geohazard_data_agent` at the curated tables, writes its operating
instructions, and publishes it so Foundry can consume it as a tool.

| | |
| --- | --- |
| **Scope** | `gold_rf1_risk_hotspots`, `gold_rf1_band_summary`, `gold_rf1_risk_matrix`, `gold_rf1_risk_pixels`, `silver_source_features`, `silver_source_coverage` |
| **Excluded** | `geometry_wkt`, `properties_json`, `geometry_json` — unbounded geometry and raw source JSON |

> `datasource.select()` selects a table with **all** of its columns; there is no
> per-column SDK call. The exclusions above and the canonical-run pinning are
> applied by `scripts/fabric/harden_data_agent.py`, which patches the item
> definition directly. Run it after this notebook or the exclusions are
> documentation only.

This runs through `fabric-data-agent-sdk`. The datasource-attachment REST API
(`/dataagents/{id}/staging/datasources`) is undocumented preview: the type enum is
`LakehouseTables`, but the item-reference field name is not discoverable from its
error messages, so the SDK is the supported path.

Idempotent — re-running re-selects the same tables and republishes.

## 1. Parameters

In [ ]:
DATA_AGENT_NAME = "geohazard_data_agent"
GOLD_LAKEHOUSE = "gold_lakehouse"
SILVER_LAKEHOUSE = "silver_lakehouse"

## 2. Inspect the SDK surface

Signatures have shifted across preview releases, so resolve them at runtime rather
than assuming. If a call below fails, this output shows what the installed version
actually offers.

In [ ]:
import inspect

from fabric.dataagent.client import FabricDataAgentManagement

print("FabricDataAgentManagement methods:")
for name, member in inspect.getmembers(FabricDataAgentManagement, inspect.isfunction):
    if name.startswith("_"):
        continue
    try:
        print(f"  {name}{inspect.signature(member)}")
    except (TypeError, ValueError):
        print(f"  {name}(...)")

agent = FabricDataAgentManagement(DATA_AGENT_NAME)
print(f"\nLoaded data agent: {DATA_AGENT_NAME}")

## 3. Table scope

Curated objects only. The hotspot table is the primary object for "where" and "why"
questions; `silver_source_features` is the sanctioned path to source detail.

In [ ]:
TABLE_SCOPE = [
    (GOLD_LAKEHOUSE, "dbo", "gold_rf1_risk_hotspots"),
    (GOLD_LAKEHOUSE, "dbo", "gold_rf1_band_summary"),
    (GOLD_LAKEHOUSE, "dbo", "gold_rf1_risk_matrix"),
    (GOLD_LAKEHOUSE, "dbo", "gold_rf1_risk_pixels"),
    (SILVER_LAKEHOUSE, "dbo", "silver_source_features"),
    (SILVER_LAKEHOUSE, "dbo", "silver_source_coverage"),
]

AGENT_INSTRUCTIONS = """
You answer questions about one completed RF-1 geohazard screening run at a time.

Filter every query by run_id and state clearly which run you used. Never mix runs in a
single answer.

If asked about trends, history, or "all my data", do not compare runs. Answer for the
canonical run and add one sentence explaining that cross-run comparison is not
meaningful here because the other runs are superseded. Never refuse outright.

The canonical run is b538ce7e-69bb-4fd1-8f00-7ba7e7fc0a0a. Use it whenever the
caller has not named a run, including for any open-ended question about "the data",
"the latest results", or "trends". Run IDs are UUIDs and carry no ordering, so never
infer which run is newest by sorting them.

These runs are SUPERSEDED and must never be used or reported:
  0da6944e-32c1-4c2f-b5a9-5b2c5666e8cf
  777d9049-b89f-427f-b6e2-a4aa3db8d192
  b50e84ae-8814-43d7-b418-17b1d050a02b
They were produced before a soil-attribute decoding fix and overstate risk severely. If
a caller names one, say it is superseded and answer for the canonical run instead.

Choosing a table:
- gold_rf1_band_summary for how much area or what share falls in each risk band.
- gold_rf1_risk_matrix for Susceptibility-by-Consequence questions.
- gold_rf1_risk_hotspots for anything about WHERE risk is concentrated or WHY. It is
  pre-ranked and pre-attributed with the dominant soil unit and drainage class,
  surficial geology unit, land cover, mean slope, and distance to the nearest mapped
  fault. Prefer it over gold_rf1_risk_pixels and cite the hs-NNN identifier.
- silver_source_features for questions about the underlying mapped source features:
  soil units, drainage, parent material, texture, geology, faults, their size and their
  distance from the AOI centre.
- silver_source_coverage for which configured sources returned data.
- gold_rf1_risk_pixels only for bounded aggregates. Never return the full pixel table
  and never select more than a deterministic top-N from it.

Ordering must be deterministic. Hotspots are already ranked: order by rank ascending.
Source features order by distance_from_aoi_km ascending, then feature_id. Always apply
a row limit.

Reporting rules:
- State the source table, the run_id, the filters applied, the row count, and units.
- Areas are km2, distances km, slope degrees, elevation metres, risk_score 1-25.
- Distinguish missing coverage from a measured zero. If silver_source_coverage marks a
  source empty or unavailable, say the data is absent; do not report it as zero hazard.
- Susceptibility blends indirect remote-sensing proxies with surveyed soil polygons.
  Only the soil survey is direct ground truth. Say so when it matters to the answer.
- This is screening information, not engineering advice. Do not give engineering
  conclusions, remediation recommendations, or site-specific assurances.
- If the data cannot answer the question, say so plainly rather than inferring.
""".strip()

print(f"{len(TABLE_SCOPE)} tables in scope across 2 lakehouses")

## 4. Attach the lakehouses

One datasource per lakehouse; tables are then selected within each.

In [ ]:
def existing_datasources():
    try:
        return list(agent.get_datasources() or [])
    except Exception as error:
        print(f"  get_datasources failed: {str(error).splitlines()[0][:160]}")
        return []


def datasource_label(datasource):
    for attribute in ("display_name", "name", "id", "datasource_name"):
        value = getattr(datasource, attribute, None)
        if value:
            return str(value)
    return repr(datasource)[:80]


wanted_lakehouses = [GOLD_LAKEHOUSE, SILVER_LAKEHOUSE]
present = {datasource_label(d) for d in existing_datasources()}
print(f"Existing datasources: {sorted(present) or 'none'}")

for lakehouse in wanted_lakehouses:
    if any(lakehouse in label for label in present):
        print(f"  {lakehouse}: already attached")
        continue
    attached = False
    # Signature has varied across SDK releases; try the documented forms in order.
    for attempt in (
        lambda: agent.add_datasource(lakehouse, type="lakehouse"),
        lambda: agent.add_datasource(name=lakehouse, type="lakehouse"),
        lambda: agent.add_datasource(lakehouse),
    ):
        try:
            attempt()
            print(f"  {lakehouse}: attached")
            attached = True
            break
        except TypeError:
            continue
        except Exception as error:
            print(f"  {lakehouse}: {str(error).splitlines()[0][:160]}")
            break
    if not attached:
        print(f"  {lakehouse}: NOT attached - check the SDK signatures printed above")

datasources = existing_datasources()
print(f"\nDatasources now: {[datasource_label(d) for d in datasources]}")

## 5. Select the tables

Selection is what actually exposes a table to the agent. Anything not selected stays
invisible to it, which is how the scope in `agent-architecture/fabric-data-agent.md`
is enforced rather than merely documented.

In [ ]:
selected_count = 0
for datasource in datasources:
    label = datasource_label(datasource)
    targets = [(schema, table) for lakehouse, schema, table in TABLE_SCOPE
               if lakehouse in label]
    if not targets:
        continue
    for schema, table in targets:
        for attempt in (
            lambda: datasource.select(schema, table),
            lambda: datasource.select(table),
        ):
            try:
                attempt()
                print(f"  {label}: selected {schema}.{table}")
                selected_count += 1
                break
            except TypeError:
                continue
            except Exception as error:
                print(f"  {label}: {schema}.{table} -> {str(error).splitlines()[0][:140]}")
                break

print(f"\nSelected {selected_count} tables")

## 6. Instructions and publish

In [ ]:
for attempt in (
    lambda: agent.update_configuration(instructions=AGENT_INSTRUCTIONS),
    lambda: agent.update_configuration(AGENT_INSTRUCTIONS),
):
    try:
        attempt()
        print("Instructions written")
        break
    except TypeError:
        continue
    except Exception as error:
        print(f"update_configuration failed: {str(error).splitlines()[0][:200]}")
        break

agent.publish()
print("Data agent published")

configuration = agent.get_configuration()
print("\nPublished configuration:")
print(configuration)

## 7. Smoke test

Ask the questions the demo will ask. A published agent that returns nothing is worse
than no agent, so this proves the wiring before anyone sees it.

In [ ]:
QUESTIONS = [
    "What percentage and area falls in each risk band?",
    "What are the top 3 ranked hotspots, and what soil drainage class underlies each?",
    "Which configured sources returned no records for this run?",
]

try:
    from fabric.dataagent.client import FabricDataAgentAPI

    client = FabricDataAgentAPI(DATA_AGENT_NAME)
    for question in QUESTIONS:
        print(f"\nQ: {question}")
        try:
            thread = client.create_thread()
            run = client.get_or_create_run(thread)
            answer = client.ask(question, thread=thread) if hasattr(client, "ask") else run
            print(f"A: {str(answer)[:800]}")
        except Exception as error:
            print(f"   (query failed: {str(error).splitlines()[0][:200]})")
except Exception as error:
    print(f"Programmatic query client unavailable ({str(error).splitlines()[0][:160]}).")
    print("Test the published agent from the Fabric portal or from Foundry instead.")